# E5-base: base vs fine-tuned (retrieval бенчмарк)

Сравнение `intfloat/multilingual-e5-base` (zero-shot) и дообученного варианта
(Augmented SBERT — CoSENTLoss на gold_train + silver).


In [16]:
# Поднимаемся к корню thesis/, чтобы относительные пути работали из подпапки
import os
from pathlib import Path
_p = Path.cwd()
while _p.name != 'thesis' and _p.parent != _p:
    _p = _p.parent
if _p.name == 'thesis':
    os.chdir(_p)
print('CWD:', Path.cwd())


CWD: c:\Users\Admin\Documents\диплом\thesis


In [17]:
import warnings
warnings.filterwarnings('ignore')

import json
import html
import time
import gc
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
from scipy import stats
from IPython.display import HTML, display

# sentence-transformers 5.x ↔ transformers 4.57 совместимость
import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import lancedb

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')


DEVICE: cpu


Файл считает три блока:

1. **Метрики на golden_eval.parquet** (1598 пар).
2. **Метрики LanceDB-retrieval** на 19 GT-парах.
3. **Визуальная таблица** top-5.

**Важно для E5:** добавляются префиксы `passage: ` (для постов) и
`query: ` (для описаний) — это делается автоматически в metrics-функциях
по полям `doc_prefix` / `query_prefix` в MODELS.


In [18]:
# ==================== МОДЕЛИ ДЛЯ СРАВНЕНИЯ ====================
# Каждая запись = одна модель в этом бенчмарке. Чтобы выключить — закомментируй.

MODELS = [
    {
        'key':          'e5_base',
        'display_name': 'E5-base zero-shot',
        'model_path':   'intfloat/multilingual-e5-base',
        'table_name':   'e5-base-base-50k',
        'doc_prefix':   'passage: ',
        'query_prefix': 'query: ',
        'color':        '#ffe5d0',
    },
    {
        'key':          'e5_ft',
        'display_name': 'E5-base fine-tuned',
        'model_path':   'models/bi-encoder-e5-finetuned',
        'table_name':   'e5-base-fine-tuned-50k',
        'doc_prefix':   'passage: ',
        'query_prefix': 'query: ',
        'color':        '#fdb675',
    },
]

# Закомментированный слот для bge-m3 fine-tuned — раскомментируй,
# когда модель и таблица 'bge-m3-fine-tuned-50k' будут готовы:
# MODELS.append({
#     'key':          'bge_ft',
#     'display_name': 'USER-bge-m3 fine-tuned',
#     'model_path':   'models/bi-encoder-bge-m3-finetuned',
#     'table_name':   'bge-m3-fine-tuned-50k',
#     'doc_prefix':   '',
#     'query_prefix': '',
#     'color':        '#8ce99a',
# })

# --------- общие параметры ---------
LANCEDB_PATH    = './lancedb_store'
EVAL_PARQUET    = 'data/golden/golden_eval.parquet'
GT_POSTS_JSON   = 'ground_truth_posts.json'
GT_PAIRS_JSON   = 'ground_truth_pairs.json'
TOP_K_VISUAL    = 5     # сколько постов показать в визуальной таблице
TOP_K_RETRIEVAL = 20    # глубина для Hit@K и MRR в LanceDB-блоке
BATCH_SIZE      = 64    # для encode на golden_eval (1598 строк)

print(f'Будет сравниваться моделей: {len(MODELS)}')
for m in MODELS:
    print(f"  - {m['display_name']:30s} ({m['table_name']})")


Будет сравниваться моделей: 2
  - E5-base zero-shot              (e5-base-base-50k)
  - E5-base fine-tuned             (e5-base-fine-tuned-50k)


In [19]:
# Проверяем, что все нужные LanceDB-таблицы существуют ДО загрузки моделей
db = lancedb.connect(LANCEDB_PATH)
available = set(db.table_names())
missing = [m for m in MODELS if m['table_name'] not in available]
if missing:
    msg = '\n'.join(f"  - {m['display_name']}: нет таблицы {m['table_name']}" for m in missing)
    raise RuntimeError(
        f'В {LANCEDB_PATH} отсутствуют таблицы для следующих моделей:\n{msg}\n\n'
        f'Сначала прогони thesis/db/create-all-dbs.ipynb для нужных моделей, '
        f'либо закомментируй их в MODELS выше.'
    )
print(f'Все {len(MODELS)} таблиц на месте.')

# Заодно проверяем golden_eval.parquet и GT-файлы
for path in [EVAL_PARQUET, GT_POSTS_JSON, GT_PAIRS_JSON]:
    assert os.path.exists(path), f'Нет файла: {path}'
print('golden_eval.parquet, ground_truth_*.json — на месте.')


Все 2 таблиц на месте.
golden_eval.parquet, ground_truth_*.json — на месте.


In [20]:
# Загружаем golden_eval.parquet и ground_truth_pairs.json
from datasets import load_dataset

eval_ds = load_dataset('parquet', data_files=EVAL_PARQUET, split='train')
descriptions = list(eval_ds['product_desc'])
posts        = list(eval_ds['post_text'])
gold_scores  = np.array(eval_ds['score'], dtype=float)
print(f'golden_eval: {len(descriptions):,} пар')
print(f'  пример desc:  {descriptions[0][:80]}...')
print(f'  пример post:  {posts[0][:80]}...')
print(f'  пример score: {gold_scores[0]:.3f}')

with open(GT_POSTS_JSON, encoding='utf-8') as f:
    gt_posts = json.load(f)
with open(GT_PAIRS_JSON, encoding='utf-8') as f:
    gt_pairs = json.load(f)
post_text_by_num = {p['post_id']: p['text'] for p in gt_posts}
print(f'GT: {len(gt_pairs)} пар, {len(gt_posts)} эталонных постов')


golden_eval: 1,598 пар
  пример desc:  Получите уникальный шанс окунуться в атмосферу любимых сериалов раньше всех! При...
  пример post:  Друзья, пока я тут помаленьку кую текстовый и видеоконтент, советую посмотреть т...
  пример score: 0.000
GT: 19 пар, 10 эталонных постов


## A. Метрики на golden_eval.parquet (1598 пар)

Кодируем все 1598 описаний и постов каждой моделью, считаем cosine на парах
(для корреляций с эталонными скорами) и cosine-матрицу (для retrieval-метрик
внутри 1598 кандидатов: каждый desc_i ищет post_i среди 1597 дистракторов).


In [21]:
# Функция: посчитать все 'статистические' метрики одной модели на golden_eval
def compute_parquet_metrics(model, model_cfg, descriptions, posts, gold_scores):
    """Возвращает dict с метриками: spearman/pearson/kendall + NDCG/MRR/Hit@K."""
    desc_in = [model_cfg['query_prefix'] + d for d in descriptions] if model_cfg['query_prefix'] else descriptions
    post_in = [model_cfg['doc_prefix']   + p for p in posts]        if model_cfg['doc_prefix']   else posts

    desc_embs = model.encode(desc_in, batch_size=BATCH_SIZE, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False)
    post_embs = model.encode(post_in, batch_size=BATCH_SIZE, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False)

    # Cosine на ПАРАХ (i, i) — это и есть предсказанный скор для (desc_i, post_i)
    pair_scores = (desc_embs * post_embs).sum(dim=1).cpu().numpy()

    # Корреляции с эталоном
    spearman_r, _ = stats.spearmanr(gold_scores, pair_scores)
    pearson_r, _  = stats.pearsonr(gold_scores, pair_scores)
    kendall_t, _  = stats.kendalltau(gold_scores, pair_scores)

    # Ranking-метрики: для каждого desc_i кандидаты — все 1598 постов;
    # истинный — post_i (релевантность = gold_scores[i]); пары с gold=0 пропускаем.
    sim_matrix = cos_sim(desc_embs, post_embs).cpu().numpy()
    n = len(descriptions)

    ndcg_at = {1: [], 3: [], 5: [], 10: []}
    rrs, hit_at = [], {1: 0, 3: 0, 5: 0, 10: 0}
    used = 0
    for i in range(n):
        if gold_scores[i] <= 0:
            continue
        used += 1
        sims = sim_matrix[i]
        order = np.argsort(-sims)
        rank = int(np.where(order == i)[0][0]) + 1
        rrs.append(1.0 / rank)
        for k in hit_at:
            if rank <= k:
                hit_at[k] += 1
        # NDCG@k: одна релевантная позиция с gain = gold_scores[i]
        for k in ndcg_at:
            if rank <= k:
                dcg = gold_scores[i] / np.log2(rank + 1)
                idcg = gold_scores[i] / np.log2(2)  # idealный — на позиции 1
                ndcg_at[k].append(dcg / idcg)
            else:
                ndcg_at[k].append(0.0)

    return {
        'spearman':   spearman_r,
        'pearson':    pearson_r,
        'kendall':    kendall_t,
        'mrr':        float(np.mean(rrs)) if rrs else 0.0,
        'hit@1':      hit_at[1] / used if used else 0.0,
        'hit@3':      hit_at[3] / used if used else 0.0,
        'hit@5':      hit_at[5] / used if used else 0.0,
        'hit@10':     hit_at[10] / used if used else 0.0,
        'ndcg@1':     float(np.mean(ndcg_at[1])),
        'ndcg@3':     float(np.mean(ndcg_at[3])),
        'ndcg@5':     float(np.mean(ndcg_at[5])),
        'ndcg@10':    float(np.mean(ndcg_at[10])),
        'eval_pairs': used,
    }


## B. Метрики LanceDB-retrieval на 19 GT парах

Реалистичный сценарий: для каждого описания товара модель ищет ответ в
**настоящем индексе из 50 000 постов**. Считаем, найдётся ли целевой пост
в top-K и на каком ранге.


In [22]:
# Функция: метрики LanceDB-retrieval на 19 GT парах для одной модели
def compute_lancedb_metrics(model, model_cfg, gt_pairs, post_text_by_num, top_k=20):
    table = db.open_table(model_cfg['table_name'])
    qprefix = model_cfg['query_prefix']

    ranks = []      # ранг целевого поста среди top_k (None — не нашёлся)
    rrs   = []
    hit   = {5: 0, 10: 0, 20: 0}
    found_count = 0

    for pair in gt_pairs:
        target_text = post_text_by_num[pair['post_num']].strip()
        q_in = (qprefix + pair['description']) if qprefix else pair['description']
        qvec = model.encode([q_in], normalize_embeddings=True)[0].tolist()
        rows = (table.search(qvec, query_type='vector')
                     .limit(top_k)
                     .select(['text', 'channel', 'category'])
                     .to_list())
        rank = None
        for i, r in enumerate(rows, 1):
            if r['text'].strip() == target_text:
                rank = i
                break
        ranks.append(rank)
        if rank is not None:
            found_count += 1
            rrs.append(1.0 / rank)
            for k in hit:
                if rank <= k:
                    hit[k] += 1
        else:
            rrs.append(0.0)

    n = len(gt_pairs)
    return {
        'ranks':     ranks,
        'mrr':       float(np.mean(rrs)),
        'hit@5':     hit[5] / n,
        'hit@10':    hit[10] / n,
        'hit@20':    hit[20] / n,
        'found':     f'{found_count}/{n}',
        'mean_rank': float(np.mean([r for r in ranks if r is not None])) if found_count else float('nan'),
    }


In [23]:
# ============================================================
# ГЛАВНЫЙ ЦИКЛ ПО МОДЕЛЯМ: загружаем, считаем оба блока метрик
# ============================================================
results = OrderedDict()      # key -> {parquet_metrics, lancedb_metrics}
loaded_models = OrderedDict() # key -> SentenceTransformer (нужен ниже для визуала)

for cfg in MODELS:
    print(f"\n--- {cfg['display_name']} ({cfg['model_path']}) ---")
    t0 = time.time()
    model = SentenceTransformer(cfg['model_path'], device=DEVICE)
    model.max_seq_length = 256
    print(f'  загружена за {time.time()-t0:.0f}с, dim={model.get_sentence_embedding_dimension()}')

    print('  parquet metrics...', end=' ', flush=True)
    t0 = time.time()
    pm = compute_parquet_metrics(model, cfg, descriptions, posts, gold_scores)
    print(f'{time.time()-t0:.0f}с')
    print(f'    Spearman={pm["spearman"]:.4f}  MRR={pm["mrr"]:.4f}  Hit@5={pm["hit@5"]:.4f}')

    print('  lancedb metrics...', end=' ', flush=True)
    t0 = time.time()
    lm = compute_lancedb_metrics(model, cfg, gt_pairs, post_text_by_num,
                                  top_k=TOP_K_RETRIEVAL)
    print(f'{time.time()-t0:.0f}с')
    print(f'    found={lm["found"]}  MRR={lm["mrr"]:.4f}  Hit@5={lm["hit@5"]:.4f}')

    results[cfg['key']] = {'cfg': cfg, 'parquet': pm, 'lancedb': lm}
    loaded_models[cfg['key']] = model

print('\nГотово.')



--- E5-base zero-shot (intfloat/multilingual-e5-base) ---
  загружена за 14с, dim=768
  parquet metrics... 242с
    Spearman=0.1490  MRR=0.0970  Hit@5=0.1422
  lancedb metrics... 

You are trying to use a model that was created with Sentence Transformers version 5.3.0, but you're currently using version 5.2.3. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


3с
    found=0/19  MRR=0.0000  Hit@5=0.0000

--- E5-base fine-tuned (models/bi-encoder-e5-finetuned) ---
  загружена за 3с, dim=768
  parquet metrics... 242с
    Spearman=0.6255  MRR=0.1043  Hit@5=0.1480
  lancedb metrics... 3с
    found=0/19  MRR=0.0000  Hit@5=0.0000

Готово.


## Сводки


In [24]:
# Сводная таблица: метрики на golden_eval.parquet
rows = []
for key, r in results.items():
    p = r['parquet']
    rows.append({
        'Модель':     r['cfg']['display_name'],
        'Spearman ρ': round(p['spearman'], 4),
        'Pearson r':  round(p['pearson'],  4),
        'Kendall τ':  round(p['kendall'],  4),
        'MRR':        round(p['mrr'],      4),
        'Hit@1':      round(p['hit@1'],    4),
        'Hit@5':      round(p['hit@5'],    4),
        'Hit@10':     round(p['hit@10'],   4),
        'NDCG@5':     round(p['ndcg@5'],   4),
        'NDCG@10':    round(p['ndcg@10'],  4),
    })
df_parquet = pd.DataFrame(rows)
print(f'Метрики на golden_eval.parquet ({results[next(iter(results))]["parquet"]["eval_pairs"]} пар с gold>0)')
display(df_parquet.style.background_gradient(cmap='YlGn',
        subset=['Spearman ρ','Pearson r','Kendall τ','MRR','Hit@1','Hit@5','Hit@10','NDCG@5','NDCG@10']))


Метрики на golden_eval.parquet (1034 пар с gold>0)


,Модель,Spearman ρ,Pearson r,Kendall τ,MRR,Hit@1,Hit@5,Hit@10,NDCG@5,NDCG@10
0,E5-base zero-shot,0.149000,0.300000,0.096500,0.097000,0.049300,0.142200,0.175000,0.099600,0.110200
1,E5-base fine-tuned,0.625500,0.617600,0.464700,0.104300,0.055100,0.148000,0.194400,0.103700,0.118700


In [25]:
# Сводная таблица: LanceDB-retrieval на 19 GT парах
rows = []
for key, r in results.items():
    l = r['lancedb']
    rows.append({
        'Модель':       r['cfg']['display_name'],
        'Found':        l['found'],
        f'Hit@5':       round(l['hit@5'],  4),
        f'Hit@10':      round(l['hit@10'], 4),
        f'Hit@20':      round(l['hit@20'], 4),
        'MRR':          round(l['mrr'],    4),
        'Mean rank':    round(l['mean_rank'], 1) if not np.isnan(l['mean_rank']) else '—',
    })
df_lancedb = pd.DataFrame(rows)
print(f'Метрики LanceDB-retrieval (поиск в индексе из 50k постов, {len(gt_pairs)} запросов)')
display(df_lancedb.style.background_gradient(cmap='YlGn',
        subset=['Hit@5','Hit@10','Hit@20','MRR']))


Метрики LanceDB-retrieval (поиск в индексе из 50k постов, 19 запросов)


,Модель,Found,Hit@5,Hit@10,Hit@20,MRR,Mean rank
0,E5-base zero-shot,0/19,0.000000,0.000000,0.000000,0.000000,—
1,E5-base fine-tuned,0/19,0.000000,0.000000,0.000000,0.000000,—


In [26]:
# Сводка: ранг целевого поста по каждой GT-паре, по каждой модели
def fmt_rank(r):
    if r is None:
        bg, bd = '#f8d7da', '#f1aeb5'; txt = '—'
    elif r <= 5:
        bg, bd = '#d4edda', '#28a745'; txt = f'#{r}'
    elif r <= 20:
        bg, bd = '#fff3cd', '#ffc107'; txt = f'#{r}'
    else:
        bg, bd = '#f8d7da', '#f1aeb5'; txt = f'#{r}'
    return (f'<span style="background:{bg};color:#000;padding:2px 8px;'
            f'border-radius:3px;font-weight:bold;border:1px solid {bd};">{txt}</span>')

def esc(s):
    return html.escape(str(s)).replace('\n', '<br>')

header_cells = ''.join(
    f'<th style="padding:8px 10px;color:#000;border-bottom:2px solid #adb5bd;background:{r["cfg"]["color"]};">'
    f'{esc(r["cfg"]["display_name"])}</th>'
    for r in results.values()
)

body_rows = []
for idx, pair in enumerate(gt_pairs, 1):
    cells = ''.join(
        f'<td style="padding:6px 10px;text-align:center;border-bottom:1px solid #e9ecef;">'
        f'{fmt_rank(r["lancedb"]["ranks"][idx-1])}</td>'
        for r in results.values()
    )
    body_rows.append(
        f'<tr style="background:#ffffff;color:#000;">'
        f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">{idx}</td>'
        f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">GT #{pair["post_num"]}</td>'
        f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">{esc(pair.get("imt_name",""))[:80]}</td>'
        f'{cells}</tr>'
    )
rows_html = ''.join(body_rows)

display(HTML(f'''
<div style="font-family:system-ui,sans-serif;margin:16px 0;color:#000;">
    <div style="font-size:15px;margin-bottom:8px;color:#000;"><b>Ранг целевого поста по каждой GT-паре</b>
    (зелёный — top-5, жёлтый — top-20, красный — за пределами top-{TOP_K_RETRIEVAL})</div>
    <table style="border-collapse:collapse;font-size:13px;background:#ffffff;color:#000;">
        <thead><tr style="background:#e9ecef;color:#000;">
            <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">#</th>
            <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">GT пост</th>
            <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">Товар</th>
            {header_cells}
        </tr></thead>
        <tbody>{rows_html}</tbody>
    </table>
</div>
'''))


#,GT пост,Товар,E5-base zero-shot,E5-base fine-tuned
1,GT #1,Затирка для плитки готовая - белая,—,—
2,GT #1,Самоклеящиеся панели для стен на кухню 60х30см пвх 15шт,—,—
3,GT #2,Развивашки 2-3-4 года/пиши стирай тетрадь/книги для малышей,—,—
4,GT #2,"Детская мозаика (5 цветов, 40 элементов) ""Кораблик""",—,—
5,GT #3,Жиросжигатель для похудения женщинам 60 капсул,—,—
6,GT #3,Таблетки для похудения - Эффективный жиросжигатель,—,—
7,GT #4,Накидка на сиденье DongFeng Fengshen Yixuan GS,—,—
8,GT #4,Кроссовер Monjaro,—,—
9,GT #5,Матрас надувной двуспальный 203х152см с подушками и насосом,—,—
10,GT #5,Гуд Найт Мягкое фито снотворное для сна,—,—


## C. Визуал: top-5 по каждой модели для каждой GT-пары

Для каждой из 19 пар — описание товара, целевой пост и таблица top-5 по
каждой модели бок-о-бок. Если целевой пост попал в top-5 у модели — её
ячейка подсвечивается зелёным.


In [27]:
# Функция: для одной GT-пары показать запрос и таблицу top-K по каждой модели
def render_pair_topk(idx, total, pair, target_text, model_topk, top_k=5):
    """model_topk: OrderedDict[key -> list of {text, channel, category} dicts]"""
    target_norm = target_text.strip()

    # Заголовки колонок
    col_headers = ''.join(
        f'<th style="padding:10px;text-align:left;background:{cfg["color"]};'
        f'border-bottom:2px solid #495057;color:#000;font-weight:bold;'
        f'border-right:1px solid #adb5bd;width:{round(100/len(model_topk), 2)}%;">'
        f'{esc(cfg["display_name"])}</th>'
        for cfg in (results[k]['cfg'] for k in model_topk.keys())
    )

    # Тело таблицы: строка на каждый ранг 1..top_k
    body = ''
    for rank in range(1, top_k + 1):
        cells = ''
        for key, rows in model_topk.items():
            if rank > len(rows):
                cells += '<td style="padding:10px;color:#000;border-right:1px solid #e9ecef;border-bottom:1px solid #e9ecef;vertical-align:top;">—</td>'
                continue
            r = rows[rank - 1]
            is_target = (r['text'].strip() == target_norm)
            bg = '#d4edda' if is_target else '#ffffff'
            border = '4px solid #28a745' if is_target else 'none'
            star = ' ★' if is_target else ''
            text_full = r['text'].replace('\n', ' ')
            cells += (
                f'<td style="padding:10px;color:#000;background:{bg};'
                f'border-left:{border};border-right:1px solid #e9ecef;'
                f'border-bottom:1px solid #e9ecef;vertical-align:top;font-size:12px;">'
                f'<div style="font-weight:bold;font-size:11px;margin-bottom:4px;color:#495057;">'
                f'#{rank}{star} · @{esc(r["channel"])} · {esc(r.get("category",""))}</div>'
                f'<div style="line-height:1.4;color:#000;">{esc(text_full)}</div>'
                f'</td>'
            )
        body += f'<tr>{cells}</tr>'

    return HTML(f'''
    <div style="border:2px solid #495057;border-radius:8px;margin:32px 0;
                background:#ffffff;font-family:system-ui,sans-serif;
                overflow:hidden;color:#000;">
        <div style="background:#e9ecef;padding:14px 20px;border-bottom:1px solid #ced4da;">
            <div style="font-size:17px;font-weight:bold;color:#000;">
                Пара {idx}/{total} · GT post #{pair["post_num"]}
            </div>
            <div style="font-size:13px;color:#495057;margin-top:4px;">
                {esc(pair.get("imt_name",""))} · {esc(pair.get("subj_name",""))}
            </div>
        </div>
        <div style="padding:14px 20px;background:#e7f3ff;border-bottom:1px solid #cfe2ff;">
            <div style="font-weight:bold;color:#000;font-size:12px;text-transform:uppercase;
                        margin-bottom:6px;letter-spacing:0.5px;">Запрос (описание товара)</div>
            <div style="color:#000;font-size:13px;line-height:1.5;">{esc(pair["description"])}</div>
        </div>
        <div style="padding:14px 20px;background:#d4edda;border-bottom:1px solid #c3e6cb;">
            <div style="font-weight:bold;color:#000;font-size:12px;text-transform:uppercase;
                        margin-bottom:6px;letter-spacing:0.5px;">Целевой пост (должен попасть в top-{top_k})</div>
            <div style="color:#000;font-size:13px;line-height:1.5;white-space:pre-wrap;">{esc(target_text)}</div>
        </div>
        <table style="width:100%;border-collapse:collapse;background:#ffffff;table-layout:fixed;">
            <thead><tr>{col_headers}</tr></thead>
            <tbody>{body}</tbody>
        </table>
    </div>
    ''')


In [28]:
# ============================================================
# ВИЗУАЛ: для каждой из 19 GT пар — топ-5 по каждой модели
# ============================================================
for idx, pair in enumerate(gt_pairs, 1):
    target_text = post_text_by_num[pair['post_num']]
    model_topk = OrderedDict()
    for cfg in MODELS:
        model = loaded_models[cfg['key']]
        table = db.open_table(cfg['table_name'])
        q_in = (cfg['query_prefix'] + pair['description']) if cfg['query_prefix'] else pair['description']
        qvec = model.encode([q_in], normalize_embeddings=True)[0].tolist()
        rows = (table.search(qvec, query_type='vector')
                     .limit(TOP_K_VISUAL)
                     .select(['text', 'channel', 'category'])
                     .to_list())
        model_topk[cfg['key']] = rows
    display(render_pair_topk(idx, len(gt_pairs), pair, target_text, model_topk, top_k=TOP_K_VISUAL))


E5-base zero-shot,E5-base fine-tuned
"#1 · @elenazimenkova · Семья и детиFaberlic Home Как же я ее ждала! В Faberlic появилась новая серия посуды из нержавеющей стали Давно хотела глубокую сковородку (артикул 910440), ведь в ней можно готовить без воды и без масла, используя собственную влагу продуктов. Влага свободно циркулирует внутри емкости, пар почти не выходит, а продукты тушатся в собственном соку Тройное термоаккумулирующее дно Подходит для всех видов плит, включая индукционные (у меня как раз индукция) Размер: диаметр – 24 см, высота – 6 см. Объем: 2,7 л.Она великолепна!Крышка стеклянная с отверстием для выхода пара, корпус с делениями для определения объема пищи, пластиковые ручки. Экологически чистая, сохраняющая витамины, экономящая газ/электричество, масло, безвредная для здоровья посуда Ну и конечно, хочется отметить стильный лаконичный внешний вид. В ней готовить одно удовольствие","#1 · @elenazimenkova · Семья и детиFaberlic Home Как же я ее ждала! В Faberlic появилась новая серия посуды из нержавеющей стали Давно хотела глубокую сковородку (артикул 910440), ведь в ней можно готовить без воды и без масла, используя собственную влагу продуктов. Влага свободно циркулирует внутри емкости, пар почти не выходит, а продукты тушатся в собственном соку Тройное термоаккумулирующее дно Подходит для всех видов плит, включая индукционные (у меня как раз индукция) Размер: диаметр – 24 см, высота – 6 см. Объем: 2,7 л.Она великолепна!Крышка стеклянная с отверстием для выхода пара, корпус с делениями для определения объема пищи, пластиковые ручки. Экологически чистая, сохраняющая витамины, экономящая газ/электричество, масло, безвредная для здоровья посуда Ну и конечно, хочется отметить стильный лаконичный внешний вид. В ней готовить одно удовольствие"
#2 · @wb_skidkamam · ПродажиПо 86₽ выходит за штуку Faberlic Карандаш пятновыводитель Фаберлик 2 шт,"#2 · @timremont1 · Интерьер и строительствоПРО ПЛЕСЕНЬ И ГРИБОК Дорогие подписчики, появление плесени в квартире-это опасное явление! Ведь это вызывает аллергию, проблемы со здоровьем, снижение иммунитета . Важно понимать, что плесень может появляться только в тёплых и влажных местах При демонтажных работах закройте все сквозные щели на улицу и обработайте поверхности грунтовкой aquastop . Никогда не спешите с ремонтом. Делайте все последовательно и давайте материалам сохнуть. Если вам пообещали ремонт квартиры под ключ за 2 - 3 месяца, то это повод задуматься. Дайте штукатурке высохнуть, прежде чем делать шпаклевку, дайте шпаклевке высохнуть, прежде чем клеить обои. Дайте стяжке высохнуть прежде чем класть Напольные покрытия Так же важно понимать, что подтеки от соседа сверху вас не спасут, если вы сделаете СЕБЕ гидроизоляцию. Нужно делать в него наверху Расскажите про свои случаи с плесенью . Спасибо за внимание всех обнял #ремонт #санузел #ремо#ремонтванной #кузня #дизайнинтерьера #дизайн #плесень #грибок"
#3 · @wb_skidkamam · ПродажиПо 86₽ выходит за штуку Faberlic Карандаш пятновыводитель Фаберлик 2 шт,"#3 · @obzor_fix_price · ПродажиБюджетный домашний скраб для тела СОХРАНИ ПОЛЕЗНЫЙ ПОСТ Основные ингредиенты:на втором фото в галерее Кофейный скраб позволит при постоянном применении заметно улучшить состояние кожи: •Улучшает микроциркуляцию крови•Повышает эластичность и тонус•Смягчает и омолаживает кожу•Способствует обновлению и регенерации клеток Использовать скраб необходимо перед другими косметическими процедурами,депиляцией Обязательно!Только на влажную кожу.Учтите этот момент Для усиления действия можно использовать мягкую щетку или мочалку. Предосторожности Не наносить на раздраженную и поврежденную кожу Не использовать перед солнечными ваннами.Потому как скраб удаляет защитный слой кожи,и она не сможет эффективно противостоять ультрафиолету Не сразу же после депиляции аллергия хотя бы на один из компонентов скраба Если твоя кожа слишком сухая,то этим средством лучше не пользоваться После скраба нужно обязательно наносить увлажняющий или защитный крем.Так же, отлично п

E5-base zero-shot,E5-base fine-tuned
"#1 · @wb_obzor_tovarov · БлогиПолка для ванной настенная пластиковая, без сверленияНабор 2 шт495₽#ванная","#1 · @plotnik_remont · Интерьер и строительствоА вы знаете из чего состоит ламинат? Ламинат состоит из 5 различных слоев, которые спрессовываются между собой под высоким давлением: 1 Защитный слой из прочной смоляной пленки; 2 Декоративный слой, выполняемый, как правило, из бумаги или мебельной фольги, рисунок которой имитирует природный материал; 3 Пленка, увеличивающая влагостойкость, расположенная в середеине этого ламинатного ""пирога""; 4 Основной слой, плита HDF, панель ДВП или ДСП из древесно-волокнистой плиты высокой степени плотности, полученной методом горячего прессования; 5 Слой из влагостойкой бумаги или пластика, пропитанный меламиновой смолой, придающий панелям жесткость и защищающий пол от влаги."
"#2 · @wb_obzor_tovarov · БлогиПолка для ванной настенная пластиковая, без сверленияНабор 2 шт495₽#ванная","#2 · @timremont1 · Интерьер и строительствоПРО ПЛЕСЕНЬ И ГРИБОК Дорогие подписчики, появление плесени в квартире-это опасное явление! Ведь это вызывает аллергию, проблемы со здоровьем, снижение иммунитета . Важно понимать, что плесень может появляться только в тёплых и влажных местах При демонтажных работах закройте все сквозные щели на улицу и обработайте поверхности грунтовкой aquastop . Никогда не спешите с ремонтом. Делайте все последовательно и давайте материалам сохнуть. Если вам пообещали ремонт квартиры под ключ за 2 - 3 месяца, то это повод задуматься. Дайте штукатурке высохнуть, прежде чем делать шпаклевку, дайте шпаклевке высохнуть, прежде чем клеить обои. Дайте стяжке высохнуть прежде чем класть Напольные покрытия Так же важно понимать, что подтеки от соседа сверху вас не спасут, если вы сделаете СЕБЕ гидроизоляцию. Нужно делать в него наверху Расскажите про свои случаи с плесенью . Спасибо за внимание всех обнял #ремонт #санузел #ремо#ремонтванной #кузня #дизайнинтерьера #дизайн #плесень #грибок"
"#3 · @timremont1 · Интерьер и строительствоПРО ПЛЕСЕНЬ И ГРИБОК Дорогие подписчики, появление плесени в квартире-это опасное явление! Ведь это вызывает аллергию, проблемы со здоровьем, снижение иммунитета . Важно понимать, что плесень может появляться только в тёплых и влажных местах При демонтажных работах закройте все сквозные щели на улицу и обработайте поверхности грунтовкой aquastop . Никогда не спешите с ремонтом. Делайте все последовательно и давайте материалам сохнуть. Если вам пообещали ремонт квартиры под ключ за 2 - 3 месяца, то это повод задуматься. Дайте штукатурке высохнуть, прежде чем делать шпаклевку, дайте шпаклевке высохнуть, прежде чем клеить обои. Дайте стяжке высохнуть прежде чем класть Напольные покрытия Так же важно понимать, что подтеки от соседа сверху вас не спасут, если вы сделаете СЕБЕ гидроизоляцию. Нужно делать в него наверху Расскажите про свои случаи с плесенью . Спасибо за внимание всех обнял #ремонт #санузел #ремо#ремонтванной #кузня #дизайнинтерьера #дизайн #плесень #грибок","#3 · @ExclusivesWB · Мода и красотаЩетка для мытья окон и стекол телескопическая#Товары_для_дома Цена: 528₽ 2 369₽ (Скидка -77%)Многофункциональная щетка для мытья окон и стекол, зеркал, пола - это незаменимый инструмент для поддержания идеальной чистоты и блеска в доме. Основным преимуществом является удобная конструкция с регулируемой телескопической ручкой, поэтому даже панорамные окна и стекла на балконе можно помыть с двух сторон - как внутри, так и снаружи. Подходит также для таких поверхностей как потолок, стены и стеклопакеты. Ссылка на товар"
"#4 · @decor_kv · Интерьер и строительствоДизайн интерьера 3 полезных мелочей для дома.Вещи, которые облегчат вашу бытовую жизнь! Легкая и тонкая стремянка Держатель для швабры Органайзер для капсул для посудомоечной машиныДекор Квартиры","#4 · @ExclusivesWB · Мода и красотаЩетка для мытья окон и стекол телескопическая#Товары_для_дома Цена: 528₽ 2 369₽ (Скидка -77%)Многофункциональная щетка для мытья окон и сте

E5-base zero-shot,E5-base fine-tuned
"#1 · @uchim_sami · Семья и детиРебёнок не хочет читать? Значит будем с ним играть! Ведущий вид деятельности дошкольника - игра. Именно через неё ребёнок должен познавать мир, обучаться и развиваться. Сюжетные игры вызывают огромный интерес у ребёнка. Поэтому я создала сказки в стихах для обучения детей чтению, для комплексного развития и качественной подготовки к школе. Летом детям не хочется заниматься, но участвовать в увлекательных приключениях, рисовать, раскрашивать, проходить лабиринты, приклеивать картинки и т.д.прямо на даче на свежем воздухе будет с радостью каждый ребёнок! Знакомство и отработка навыка чтения с каждой буквой будет проходить в виде увлекательной сказки в стихах. По ходу развития её сюжета ребёнок будет помогать героям, проходить через препятствия, путешествовать вместе с ними, а самое главное: знакомиться с буквами, учиться читать и развиваться.Дети ждут с нетерпением каждую следующую сказку! Они твердо усваивают - учиться интересно! Это и есть залог успеха!Переходите в интернет-магазин и покупайте сказки, которые сделают детство счастливым, а взрослую жизнь - успешной!","#1 · @uchim_sami · Семья и детиРебёнок не хочет читать? Значит будем с ним играть! Ведущий вид деятельности дошкольника - игра. Именно через неё ребёнок должен познавать мир, обучаться и развиваться. Сюжетные игры вызывают огромный интерес у ребёнка. Поэтому я создала сказки в стихах для обучения детей чтению, для комплексного развития и качественной подготовки к школе. Летом детям не хочется заниматься, но участвовать в увлекательных приключениях, рисовать, раскрашивать, проходить лабиринты, приклеивать картинки и т.д.прямо на даче на свежем воздухе будет с радостью каждый ребёнок! Знакомство и отработка навыка чтения с каждой буквой будет проходить в виде увлекательной сказки в стихах. По ходу развития её сюжета ребёнок будет помогать героям, проходить через препятствия, путешествовать вместе с ними, а самое главное: знакомиться с буквами, учиться читать и развиваться.Дети ждут с нетерпением каждую следующую сказку! Они твердо усваивают - учиться интересно! Это и есть залог успеха!Переходите в интернет-магазин и покупайте сказки, которые сделают детство счастливым, а взрослую жизнь - успешной!"
"#2 · @Moms_Secrets_ru · Семья и детиЕсли эта книга попадет в руки ребенка, он узнает некоторые секреты взрослых, сможет увидеть, что они обычные люди, что им тоже присущи слабости и что под их запретами и требованиями живут настоящие чувства. Если книгу прочтет родитель, она поможет ему найти слова, чтобы поговорить с ребенком о разных сложных ситуациях и быть лучше понятым самому и лучше понять ребенка. Если же книга достанется психологу, то у него появится отличный инструмент для прояснения переживаний и обсуждения сложных отношений и событий семейной жизни и семейной истории с детьми и родителями.Психология для родителей","#2 · @banda_u · Семья и детиГрафомоторные лабиринты для тренировки рук + Шаблон -----------------------------------Такие лабиринты отлично развивают мелкую моторику, координацию движений и тренируют внимание Нарисуйте симметричные линии (или воспользуйтесь нашим шаблоном) и пусть ребёнок попробует их пройти обеими руками одновременно. Будет непросто!Если так играть ежедневно, то можно превратиться в настоящего амбидекстра Так называют людей, которые одинаково хорошо владеют обеими руками. Не обязательно входить в 1% людей с врождённой амбидекстрией, овладеть «обоерукостью» можно в любом возрасте Ещё больше умных заданий на развитие мелкой моторики и координации в наших тетрадях «Реши-Пиши» На Wildberries Ozon"
"#3 · @uchim_sami · Семья и детиВаш ребёнок знает секретики буквы Ы?Эта буква не простая и требует особого внимания. Деткам бывает сложно её запомнить. Но только не тем, которые занимаются по моим книгам! СКАЧАЙТЕ БЕСПЛАТНО ПРОДОЛЖЕНИЕ ПРОПИСИ!Ваш ребёнок узнает секреты буквы Ы, научится её писать. А так же научится писать слоги, слова и предложения с этой буквой. Для лучшего

E5-base zero-shot,E5-base fine-tuned
"#1 · @katyabux · БлогиНе могу пройти мимо всяких интересных штучек для детей. Смотрите , что нашла в Фаберлик- детская мыльная краска для купания. Взяла все цвета, что были в наличии, это зеленая краска, розовая и желтая) Еще и с сочными фруктовыми ароматами - клубника, ананас и банан. Такая пенка позволяет ребенку рисовать и проявлять фантазию прямо в ванной, и создает легкую пенку)",#1 · @manickure · Мода и красотаНе знаете что подарить на 23 февраля или 8 марта?!СЮРПРИЗ БОКС от НАШЕГО КАНАЛА Наполнение на ваш вкус и по вашему запросу! Действительно антикварные и винтажные предметы стоимостью до 100.000₽ Для заказа просто опишите интересы человека для кого предназначен подарок и ждите ярких эмоций в праздничный вечер От себя мы гарантируем:•доставка без опозданий прямо в день праздника (или раньше ) •курьерская доставка по всей России •действительно оригинальная упаковка подарка и предмет который точно понравится вашему близкому человеку Стоимость 500₽ за бокс МИНИ 2990 ЗА СТАНДАРТ7499 ЗА МАКСИМАЛЬНУЮ ВЕРСИЮ Контакты для заказа *При оформлении напишите ПРОМОКОД сюрпризбокс24И получите бесплатную доставку по всей России
"#2 · @katyabux · БлогиНе могу пройти мимо всяких интересных штучек для детей. Смотрите , что нашла в Фаберлик- детская мыльная краска для купания. Взяла все цвета, что были в наличии, это зеленая краска, розовая и желтая) Еще и с сочными фруктовыми ароматами - клубника, ананас и банан. Такая пенка позволяет ребенку рисовать и проявлять фантазию прямо в ванной, и создает легкую пенку)","#2 · @sonya_gda · Еда и кулинарияЗнаю, что многие из Вас мечтают научиться искусству оформления бранчей и фуршетов.Поэтому, в предверии ярких новогодних праздников, где застолья и подарки для родных и близких - самые главные хлопоты, я приглашаю Вас на мой ДВУХДНЕВНЫЙ ОНЛАЙН МАСТЕР-КЛАСС 7 и 8 ДЕКАБРЯ В первый день мы: Соберём бранч-боксы в эфире и поговорим о их разнообразии; Упакуем готовые боксы и разберём декор; Прямо в эфире мы попробуем продать наши шедевры и посмотрим, сколько можно заработать на одном бранч-боксе, особенно накануне праздников. На 2-ой день Я расскажу и покажу Вам, как грамотно и при этом стильно засервировать фуршет; Какие нюансы я учитываю и какие лайфхаки помогают избегать ошибок; Где искать первых клиентов и почему сегодня фуршет - это тренд на любом празднике.ДЛЯ ТЕХ, КТО ПРОВЕДЁТ С НАМИ ОБА ДНЯ МАСТЕР-КЛАССА ДО КОНЦА МЫ ПОДГОТОВИЛИ ВИДЕОУРОК ИЗ МОЕГО КУРСА ПО БРАНЧАМ Подробности по ссылке регистрируйтесь и ставьте напоминание!"
"#3 · @facktoriym · ПозновательноеЯркий и необычный подарок, который понравится абсолютно всем Развивает воображение, мелкую моторику и усидчивость! Увлекает надолго как взрослого, так и ребенка. Готовая работа классно впишется в любой интерьер ЦЕНА: 815 руб.","#3 · @workathomerus · КарьераТребуется ИллюстраторЗадачи: иллюстрации для наклеек, мерча, открытки под разные праздники, Возможно портретные иллюстрации для открыток и стикеров.Что важно:— Быстро и вовремя выполнять задачу. Уметь планировать своё время.— Быть адекватным, уметь воспринимать критику.— Быть внимательным, учитывать все комментарии.— Уметь работать в разных стилях, иметь чувство вкуса.— Креативное мышление. Уметь предложить свой вариант, если с идеями туго.— Если это портрет, то человек должен быть похож на себя. Понимать пропорции тела.Ставка от 350 ₽ в час. Может быть выше, в зависимости от опыта/скорости отрисовки. Например если весь процесс отрисовки стабильно будет занимать 2 часа → можем поднять ставку до 500 ₽.Связаться и отправить портфолио kizimmaxim -agency.ru"
"#4 · @facktoriym · ПозновательноеЯркий и необычный подарок, который понравится абсолютно всем Развивает воображение, мелкую моторику и усидчивость! Увлекает надолго как взрослого, так и ребенка. Готовая работа классно впишется в любой интерьер ЦЕНА: 815 руб.","#4 · @logopedavdom · Семья и дети​Идеи для занятий на световом столе Песочная анимация. Для рисования песком можно использовать не только кварц

E5-base zero-shot,E5-base fine-tuned
"#1 · @lit33m1d · Здоровье и фитнесРаботяги, для тех кому по каким-либо причинам не хватает сил на тренировках - к вашему вниманию предтрены от RECKFUL. Годная вещь, которую нужно использовать с умом и не борщить.Также в наличии у парней имеются как стандартные энергетики/стимуляторы так и пампилки для мощной прокачки без нагрузки для ЦНС (для тех кому хватает энергии на тренировке )WILDBERRIESOZON","#1 · @vstalaipohla · Мода и красотаФантастическая четверка против целлюльных нападений:Ролл МФР — болезненный, но с помощью упражнений можно эффективно и быстро раскатать бугристые застоиАппликатор Кузнецова — самый простой из всех. Просто выйти распаренной из душа, намазаться антицеллюльным кремом и сесть на игольчатый коврик минут на 15FatSecret - классное приложение для подсчета калорий и контроля КБЖУ. Потому что это самый эффективный способ наладить питание и контролировать его, а без питания остальные пункты могут не помочьБанки для вакуумного массажа — самый трудозатратный, но очень эффективный. Делать по кокосовому маслу круговыми движениями, пока бедра не начнут гореть. Лучше брать крупные, мягкие банки, чтобы не налепить синяковЧто добавили бы или какой больше предпочитаете? Делитесь своими способами"
"#2 · @lit33m1d · Здоровье и фитнесРаботяги, для тех кому по каким-либо причинам не хватает сил на тренировках - к вашему вниманию предтрены от RECKFUL. Годная вещь, которую нужно использовать с умом и не борщить.Также в наличии у парней имеются как стандартные энергетики/стимуляторы так и пампилки для мощной прокачки без нагрузки для ЦНС (для тех кому хватает энергии на тренировке )WILDBERRIESOZON","#2 · @leya_leyaaa · Мода и красотаПоследнее время моя активность снизилась, поэтому я срочно оформила заказ у уже любимых и проверенных Re-Feel и заказала для себя целых три коробки полезных стиков, то-есть витаминов, которые способствуют не только укреплению иммунитета, но и работают на влияние стресса, укрепляют мышцы, восполняют все нужные нам в организме функции! Давайте расскажу что же я приобрела и для чего это собственно нужноМорской коллаген с клубникой в стикахВсе мы знаем, что коллаген-это бустер для здоровья кожи, волос,костей и мышц, поэтому я срочно заказала именно эту коробочку чтобы значительно повысить свое здоровье и укрепить иммунитет Матча-латте Ее я взяла для того, чтобы усилить эффект сияющей кожи.В составе так же присутствует коллаген в формате пептидов, который усваивается быстрее и на 100% делает кожу более упругой, ногти крепкими а волосы блестящими Так же в составе присутствует пребиотик инулин, который поддерживает ЖКТ И фиолетовая коробочка, это у нас ассорти из матчи, кофе, какао и чая Матча с коллагеном, кофе с пребиотиком, пряный чай латте для иммунитета и антистресс-какое для расслабления Будем восстанавливать силы вкусно и полезно"
#3 · @sovetov_100 · ПозновательноеЧУДО СРЕДСТВО СВОИМИ РУКАМИ .Пригодится всем Универсальное средство на основе трёх компонентов Отлично справляется с жиром и застарелыми пятнами! Экологически чистая альтернатива химии.Смотрите видео со звуком,"#3 · @pavelbaranov_md · МедицинаРекомендации по теме здорового долголетия часто подчёркивают важность аэробных упражнений. Появляющиеся данные всё больше показывают, что силовые тренировки бывают столь же эффективны в этом отношении. А ещё лучше – их комбинация. В ACSM's Health & Fitness Journal вышла новая работа о важности занятий с отягощениями для здоровья в любом возрасте.Специалистам рекомендую изучить. На схеме изображены эффекты от занятий аэробными или силовыми тренировками в отношении субклеточных процессов и физиологических систем, на которые они влияют: кардиореспираторное здоровье, здоровье сосудов, окислительный потенциал мышц, психическое здоровье, мышечная сила и работоспособность.В нижней части показано, что профилактическое влияние аэробных и силовых нагрузок имеет больше общего, чем различий. К последствиям, вероятность наступления которых можно снизить за счёт регулярных 

...

E5-base zero-shot,E5-base fine-tuned
"#1 · @favouriteWB24 · Мода и красотаС такой подушкой каждое утро будет добрым!Подушка с эффектом памяти Поможет расслабить мышцы шеи и спины и занять правильное положение во время сна. Чехол снимается, а два валика помогут подобрать комфортную высоту подушки под себя. Цена сейчас: 1869₽ Обычная цена: 7331₽ Успей заказать","#1 · @alicozy · Интерьер и строительствоПорадуйте близких натуральным и стильным текстилем. Собрали для вас несколько идей подарков от бренда UTROХалат из объёмной вафлиИдеален не только для душа или ванны, но и для неторопливого утра или тёплого вечера дома. Подойдёт как мужчинам, так и женщинам.Постельное бельёНа выбор 3 линейки из разных тканей. Изящный и шелковистый тенсель, уютный варёный хлопок и практичный сатин. Все комплекты выполнены в однотонных пастельных тонах, поэтому вариант можно подобрать под любой интерьер.Шерстяной пледПледы выполнены из супертонкой овечьей шерсти. Такой плед подарит уют и тепло вашему близкому человеку. Подушка с эффектом памятиПринимает форму головы и шеи, тем самым, обеспечивая комфорт, правильное положение и мягкость во время сна. Подарок, который оценит каждый.До конца года у интернет-магазина действует скидка - 10% по промокоду ДАРИУЮТ. Если оформить заказ в ближайшие 3 дня, успеете получить его до Нового года Халаты, постельное бельё, пледы, подушка Смотреть все товары бренда"
"#2 · @ostrovok_travel · ПутешествияЗимние путешествия порой наталкивают на бесконечную череду мыслей «слишком холодно», «жарко», «надо было всё-таки взять джемпер» и «зачем я надел тёплый пуховик»... Вместе с нашими друзьями из TJ Collection делимся советами, что положить в чемодан, чтобы и ощущать себя комфортно, и при этом не отстать от моды!Реклама. ООО «КЛ ГРУПП». Erid: 2VtzqvbM2cj","#2 · @beautyjars · Мода и красотаСуперкосметика для всехИногда хочется, как в сказке — чтоб последствия рабочих стрессов, палящего солнца и веселых бессонных ночей никак не сказывались на коже. Бренд SUPERBANKA отлично подходит для людей, живущих активной жизнью в мегаполисах.СRÈME DE LA СRÈME — увлажняющий крем, который встраивается в структуру кожи и действует мягко, но эффективно. Формула заряжена на SOS-увлажнение и подходит для ежедневного ухода. Восстанавливает защитный барьер и выравнивает тон лица. Подходит как для сухой, так и для жирной кожи.FANTOMAZ — питательный зеленый тинт, меняющий цвет на один из оттенков розового, подстраиваясь под pH губ. Содержит три вида масел и особый комплекс пептидов для максимального увлажнения. Визуально увеличивает объем и держится целый день.Также в линейке средств представлены витаминная сыворотка и мицеллярный лосьон. А на Ozon сейчас действуют скидки на всю продукцию, кроме сыворотки — чем не повод обновить косметичку?Не обязательно быть триатлетом, чтобы день изо дня справляться с насыщенностью жизни на тех же уровнях сложности. Хорошо, когда о коже позаботятся за тебя. Это мы берем"
"#3 · @valberis_odezhda1 · Мода и красотаЗабудьте о холодной и неприятной зиме с этой прекрасной шапкой. Неважно, идете ли вы на прогулку по зимнему парку или отправляетесь на работу, шапка обеспечит вам комфорт и тепло на протяжении всего дня. Вы можете быть уверены, что ваша голова будет защищена от суровых погодных условий. Утеплиться","#3 · @skdesigntg · Интерьер и строительствоДля тех, кто хочет порадовать себя и близких к Новому Году, напоминаем о нашей акции: -15% при покупке 2-х кресел SPIN. Кресло SPIN — наш бесспорный бестселлер: плавные линии силуэта, мягкость, подобная облаку, и трендовый дизайн. Будьте уверены: эта модель точно впишется в интерьер и прекрасно дополнит композицию мягкой мебели в гостиной.На выбор мы предлагаем вам несколько вариантов текстиля:• эффектный жаккард ""гусиная лапка"" • нежный велюр• натуральная кожа• трендовая обивка букле Скорее к покупкам:"
#4 · @wb_ozon_sale_skidki · Мода и красотаПодушкаЦена: 656₽ (вместо 4 005₽)#товарыдо1000 Анатомическая подушка 50х70 сочетает в себе функциональность и комфорт. Она

E5-base zero-shot,E5-base fine-tuned
"#1 · @ostrovok_travel · ПутешествияЗимние путешествия порой наталкивают на бесконечную череду мыслей «слишком холодно», «жарко», «надо было всё-таки взять джемпер» и «зачем я надел тёплый пуховик»... Вместе с нашими друзьями из TJ Collection делимся советами, что положить в чемодан, чтобы и ощущать себя комфортно, и при этом не отстать от моды!Реклама. ООО «КЛ ГРУПП». Erid: 2VtzqvbM2cj","#1 · @stylistkatiakida · Мода и красотаПодборка обуви и сумок бренда Salamander. На сайте представлен широкий ассортимент женской и мужской обуви из натуральных материалов, а также аксессуаров Мои фавориты:- Базовые кроссовки из натуральной кожи за 6,5тр размеры 35-40 - Идеальные лодочки на миниатюрной шпильке - Классные чёрные лоферы из натуральной кожи тоже в р-р 35-40 Отдельно ссылками отмечу категории:• Новинки • Кеды и кроссовки • Туфли • Сумки Кстати, в рамках глобального ребрендинга по всему миру у Salamander обновился лого и фирменный стиль. Обновления коснулись и самой коллекции, в которой появились свежие современные тренды. Подписывайтесь на телеграм-канал и сохраняйте ссылку на интернет-магазин бренда"
#2 · @thecookiss · ИгрыДержите файл мода на турецкие машины!Türk arabaları için bir moda dosyası tutun! Lütfen abone olun ve ikinci bölüm yayınlanacaktır! Подписаться | Subscribe,"#2 · @SumkiBijuteryotRuslana · Мода и красотаДорожная сумка-это идеальный выбор не только для любителей спорта.Её можно взять с собой не только на тренировку в зал или бассейн,но и в путешествие. Она идеальна для похода в спортзал или для поездки, в качестве ручной клади в самолёт.Непромокаемая сумка изготовлена из качественной водонепроницаемой ткани, что обеспечивает долговечность и защиту ваших вещей.Большая сумка также оснащена удобными ручками и регулируемыми плечевым ремнём для комфортного ношения через плечо в дороге.Опт 500₽ роз 600₽Размер 48#30#23"
#3 · @thecookiss · ИгрыДержите файл мода на турецкие машины!Türk arabaları için bir moda dosyası tutun! Lütfen abone olun ve ikinci bölüm yayınlanacaktır! Подписаться | Subscribe,"#3 · @SumkiBijuteryotRuslana · Мода и красотаДорожная сумка-это идеальный выбор не только для любителей спорта.Её можно взять с собой не только на тренировку в зал или бассейн,но и в путешествие. Она идеальна для похода в спортзал или для поездки, в качестве ручной клади в самолёт.Непромокаемая сумка изготовлена из качественной водонепроницаемой ткани, что обеспечивает долговечность и защиту ваших вещей.Большая сумка также оснащена удобными ручками и регулируемыми плечевым ремнём для комфортного ношения через плечо в дороге.Опт 500₽ роз 600₽Размер 48#30#23"
"#4 · @alyatrend_home · Интерьер и строительствоЧЕХЛЫ НА ПОДУШКИ ОТ 555РОсень для меня, и я уверена, для многих из вас, то время, когда хочется максимального уюта! А уют дома напрямую зависит от текстиля — на прошлой неделе смотрели с вами классные пледы, а сегодня, думаю, самое время для чехлов на подушки Выбирала модели в более базовых цветах, но с необычным и оригинальным декором На все позиции есть реальные отзывы с фото, а на чехлы 4 и 6 есть ещё и бесплатный возврат С ромашкамиОднотонный под бархат С объемным пушистым декоромИнтересной вязки с кисточкамиЧерно-белыйОднотонный с кисточками","#4 · @SumkiBijuteryotRuslana · Мода и красотаДорожная сумка-это идеальный выбор не только для любителей спорта.Её можно взять с собой не только на тренировку в зал или бассейн,но и в путешествие. Она идеальна для похода в спортзал или для поездки, в качестве ручной клади в самолёт.Непромокаемая сумка изготовлена из качественной водонепроницаемой ткани, что обеспечивает долговечность и защиту ваших вещей.У неё вместительное внутреннее отделение для хранения спортивной формы и сменной одежды, отдельный боковой карман для обуви, а также наружные и внутренние карманы для размещения аксессуаров . Большая сумка также оснащена удобными ручками и регулируемыми плечевым ремнём для комфортного ношения через плечо в дороге.Опт 500₽ роз 600₽Размер 45#27#20"


E5-base zero-shot,E5-base fine-tuned
"#1 · @AndreyPitertsov · ПриродаПрошло уже два полноценных сезона, как я активно гоняю крупные приманки спиннинговым комплектом. Палочки в моих руках вы могли видеть разные: Хайрон до 140 или 200 г. Хелл Хаунд до 160... А вот мясорубка всегда одна - Твин 20 года 4000pg.Знаю, что многим интересно как она? Что с ней случилось после достаточно продолжительных нагрузок, не развалилась ли?К удивлению для многих скажу, что с ней все хорошо. Ход стал немного более грубым - это единственное что произошло, и это нормально. Больше и добавить нечего. Громыхать, урчать, скрипеть не начала.Так что практикой удалось подтвердить то, что ""мясорубочный"" комплект отлично подходит для больших приманок. Далее каждый сам решает на что ему ловить приятнее и комфортнее (мульт или мясорубка).",#1 · @danludan_real · БукмекерствоПОКУПАЮ БОНУСЫ STICKY BANDITS 3 MOST WANTED DANLUDAN МАКСБЕТ НОВЫЙ СЛОТ ОТ QUICKSPIN!!!ВСЕМ ПРИЯТНОГО ПРОСМОТРА!!!
"#2 · @bcodenews · ЭкономикаВысококачественная твердая дробь обеспечивает отличную резкость боя – не менее 4 диаметров (на 35 м).Использование качественных порохов гарантирует сохранение чистоты ствола при стрельбе и стабильность работы в экстремальных температурных условиях.Патроны IGLA отвечают высочайшим требованиям мировых стандартов, обеспечивая комфортную стрельбу и надежность поражения цели.#РекламаАО ""Технодинамика"" ИНН: 7719265496ERID: Kra23m6Zh",#2 · @danludan_real · БукмекерствоПОКУПАЮ БОНУСЫ STICKY BANDITS 3 MOST WANTED DANLUDAN МАКСБЕТ НОВЫЙ СЛОТ ОТ QUICKSPIN!!!ВСЕМ ПРИЯТНОГО ПРОСМОТРА!!!
"#3 · @bitmejkerskaya · Музыка​​Ребята, напоминаю вам, что мы недавно открыли Приватный канал [Битмейкерская Secret].Там мы публикуем море различного стаффчика (да-да, контент постоянно пополняется) и за скромные 690р ты сможешь целый год радовать себя качественным контентом, получать знания и, что самое главное - развиваться!Заинтересовало? Просмотреть содержимое канала ты можешь тутили жеКупить доступ в приватный канал тут","#3 · @ribalka_sovet · ПозновательноеТЕХАССКАЯ ОСНАСТКА — ИЗГОТОВЛЕНИЕ И ТЕХНИКА ЛОВЛИ Техасская оснастка относится к разряду абсолютных «незацепляек» и своим происхождением обязана одному из озер в штате Техас. Изначально оснастка применялась для ловли американского басса, но в силу своей универсальности и уловистости она завоевала популярность и у наших спиннингистов. В условиях наших водоемов Техасская оснастка с успехом применяется для ловли щуки, окуня, судака и даже язя. Что представляет собой Техасская оснастка и где применяется? Техасская оснастка устроена достаточно просто — офсетный крючок с насаженным на него силиконовым червем привязан к основной леске, на которой скользит грузило в виде пули. Между крючком и пулей устанавливается бусинка (бисер). Хотя ее наличие не всегда обязательно, но об этом позже."
"#4 · @renata_ali_shopping · Мода и красотаНу что ж образ парижанки все таки удалось примерить... и я в полном восторге Все таки дизайнеры Sèzane знают толк в принтах и как их обыграть Свитер 3500₽ оригиналом,все бирки есть,обратите внимание на скрин с оф.сайта - состав и страна производства made in china,вот так и французкий шик Свитер очень мягкий,теплый,но не жаркий,идеальный по составу,даже приятно колется. На 88/66 взяла L.Юбка 2800₽ бирок нет,состав ПЭ, очень качественного ПЭ аккуратный пошив и крой очень похожий на оригинал,рисунок тоже сходится. Мне понравилась,в уходе будет не капризной . Теперь я постоянный гость этого магазина,присмотрела ещё другие свитера Вот так Sèzane с АлиЭкспресс,кто бы мог подумать...","#4 · @optbijuteria · Мода и красотаНА КАНАЛЕ БИЖУТЕРИИ СДЕЛАЛИ РАССЫЛКУ НА РЕМНИ YSLVALENTINO CUCCIHERMES DIORLVCELINEШИКАРНЫЕ МОДЕЛИИИ ИХ РАСКУПАЮТ С БЕШЕНОЙ СКОРОСТЬЮУСПЕЙТЕ В ЛУЧШЕМ КАЧЕСТВЕ , МЫ ОЧЕНЬ ДОВОЛЬНЫ"
#5 · @MaliBoo325 · БукмекерствоКОТИКИ CatCasino 325% к депозиту . Промокод : Mal200 200% к депозиту +100FS,"#5 · @optbijuteria · Мода и красотаНА КАНАЛЕ БИЖУТЕРИИ СДЕЛАЛИ РАССЫЛКУ НА РЕМНИ YSLVALENTINO CUCCIHERMES DI

E5-base zero-shot,E5-base fine-tuned
"#1 · @AndreyPitertsov · ПриродаПрошло уже два полноценных сезона, как я активно гоняю крупные приманки спиннинговым комплектом. Палочки в моих руках вы могли видеть разные: Хайрон до 140 или 200 г. Хелл Хаунд до 160... А вот мясорубка всегда одна - Твин 20 года 4000pg.Знаю, что многим интересно как она? Что с ней случилось после достаточно продолжительных нагрузок, не развалилась ли?К удивлению для многих скажу, что с ней все хорошо. Ход стал немного более грубым - это единственное что произошло, и это нормально. Больше и добавить нечего. Громыхать, урчать, скрипеть не начала.Так что практикой удалось подтвердить то, что ""мясорубочный"" комплект отлично подходит для больших приманок. Далее каждый сам решает на что ему ловить приятнее и комфортнее (мульт или мясорубка).","#1 · @rezatribu · ПозновательноеFeeder Gum Это отрезок резины, который создает амортизацию и гасит резкие рывки рыбы.Вяжется он между фидерной оснасткой и очень тонким флюрокарбоновым поводком (например, 0.1 мм).Используется длина 10-12 см и толщина 0.6-0.8 мм.Если требуется деликатное вываживание при очень тонкой леске (0.1 мм), то применяют именно Feeder Gum."
"#2 · @rezatribu · ПозновательноеFeeder Gum Это отрезок резины, который создает амортизацию и гасит резкие рывки рыбы.Вяжется он между фидерной оснасткой и очень тонким флюрокарбоновым поводком (например, 0.1 мм).Используется длина 10-12 см и толщина 0.6-0.8 мм.Если требуется деликатное вываживание при очень тонкой леске (0.1 мм), то применяют именно Feeder Gum.","#2 · @rezatribu · ПозновательноеFeeder Gum Это отрезок резины, который создает амортизацию и гасит резкие рывки рыбы.Вяжется он между фидерной оснасткой и очень тонким флюрокарбоновым поводком (например, 0.1 мм).Используется длина 10-12 см и толщина 0.6-0.8 мм.Если требуется деликатное вываживание при очень тонкой леске (0.1 мм), то применяют именно Feeder Gum."
"#3 · @rezatribu · ПозновательноеFeeder Gum Это отрезок резины, который создает амортизацию и гасит резкие рывки рыбы.Вяжется он между фидерной оснасткой и очень тонким флюрокарбоновым поводком (например, 0.1 мм).Используется длина 10-12 см и толщина 0.6-0.8 мм.Если требуется деликатное вываживание при очень тонкой леске (0.1 мм), то применяют именно Feeder Gum.","#3 · @ribalka_sovet · ПозновательноеТЕХАССКАЯ ОСНАСТКА — ИЗГОТОВЛЕНИЕ И ТЕХНИКА ЛОВЛИ Техасская оснастка относится к разряду абсолютных «незацепляек» и своим происхождением обязана одному из озер в штате Техас. Изначально оснастка применялась для ловли американского басса, но в силу своей универсальности и уловистости она завоевала популярность и у наших спиннингистов. В условиях наших водоемов Техасская оснастка с успехом применяется для ловли щуки, окуня, судака и даже язя. Что представляет собой Техасская оснастка и где применяется? Техасская оснастка устроена достаточно просто — офсетный крючок с насаженным на него силиконовым червем привязан к основной леске, на которой скользит грузило в виде пули. Между крючком и пулей устанавливается бусинка (бисер). Хотя ее наличие не всегда обязательно, но об этом позже."
"#4 · @Pashu_aboutbusiness · БлогиНАША ВЕЛИКАЯ ПРИРОДА: КАК ЖЕ КРАСИВА! ЧТОБЫ ЛЮБИТЬ ЖИЗНЬ, НАДО ПУТЕШЕСТВОВАТЬ, ПРИЧЕМ ПО РАЗНОМУ: И КРАСИВО И RAW. НАСМОТРЕННОСТЬ, НАСЛУШЕННОСТЬ, НАЧИТАННОСТЬ ДАЁТ ВОЗМОЖНОСТЬ ВИДЕТЬ ШИРЕ, МЫСЛИТЬ КРЕАТИВНЕЕ, ЖЕЛАТЬ ЛУЧШЕГО, ДУМАТЬ ПО КРУПНОМУ, БЫТЬ БОЛЬШЕ. КТО-ТО ОБЯЗАТЕЛЬНО НАПИШЕТ, НА ЧТО? ПОНИМАЮ! НО ИМЕННО ЭТА ЦЕПОЧКА РАБОТАЕТ НЕ С КОНЦА, А С НАЧАЛА: ПРЕДСТАВЛЯТЬ, ХОТЕТЬ, ПЛАНИРОВАТЬ, ДЕЙСТВОВАТЬ, ДОСТИГАТЬ, НАСЛАЖДАТЬСЯ, ПРИВЫКАТЬ К МИНИМУМУ, И ПО НАРАСТАЮЩЕЙ. НА КАЖДОМ КРУГУ ОПРЕДЕЛИТЬ ЦЕПОЧКУ ТЕХ, С КЕМ НАЧАТЬ ДЕЛИТЬСЯ ОПЫТОМ, ПОМОЩЬЮ, ПОДДЕРЖКОЙ И ТД. НУ И ГЛАВНОЕ НЕ НЕСТИ НЕГАТИВ И ПЛОХИЕ ЭМОЦИИ, ЗА ИСКЛЮЧЕНИЕМ, КОГДА МЕШАЮТ И ВРЕДЯТ ВАМ. НУ И ЖЕЛАТЕЛЬНО НЕСТИ ДОБАВОЧНУЮ СТОИМОСТЬ В МИР И ЛЮДЯМ. Я БЫ НДС ЗАМЕНИЛ НА ПДС (Поддержка за добавочную стоимость LETS GO","#4 · @AndreyPitertsov · ПриродаВ прошлом посте обсуждали оптимальное о

E5-base zero-shot,E5-base fine-tuned
"#1 · @Salemeplease · Мода и красотаZARINA переосмыслила стиль apres ski и взяла от него лучшее: комфорт и функциональность в сочетании с кэжуал-шиком в новой лимитированной коллекции. Пуховик с эффектом металлик и костюм из фактурной вязки, белоснежная шуба с длинным мехом и брюки прямого кроя, джемпер с узором замёрзшего льда… Посмотреть всю коллекцию можно уже на сайте По промокоду SALEME скидка -15% в интернет-магазине.","#1 · @Salemeplease · Мода и красотаZARINA переосмыслила стиль apres ski и взяла от него лучшее: комфорт и функциональность в сочетании с кэжуал-шиком в новой лимитированной коллекции. Пуховик с эффектом металлик и костюм из фактурной вязки, белоснежная шуба с длинным мехом и брюки прямого кроя, джемпер с узором замёрзшего льда… Посмотреть всю коллекцию можно уже на сайте По промокоду SALEME скидка -15% в интернет-магазине."
"#2 · @valberis_odezhda1 · Мода и красотаЗабудьте о холодной и неприятной зиме с этой прекрасной шапкой. Неважно, идете ли вы на прогулку по зимнему парку или отправляетесь на работу, шапка обеспечит вам комфорт и тепло на протяжении всего дня. Вы можете быть уверены, что ваша голова будет защищена от суровых погодных условий. Утеплиться","#2 · @dnevnikox · БлогиЮбка шорты на все случаи жизни.Модель из нашего магазина на ВБ.Базовая юбка-шорты отлично сидит на фигуре, скрывает недостатки и подчеркивает вашу фигуры.Для вечеринок, прогулок, на пляж и вообще на все случаи жизни подойдет идеально.Сейчас с огромной скидкой на ВБ .Успейте заказать по вкусной цене."
"#3 · @zolotoy585club · ПродажиЭто база, девочки Даже если попросить стилиста назвать только один аксессуар, который на все 100% подойдет к любому образу, им точно станет цепочка – модная, акцентная, с интересным плетением Ставьте и оцените украшение от 1 до 5 в комментариях! Цепь, золото 585 проба, плетение Веревка.*Цены актуальны на момент публикации поста.Реклама. ООО «Регент Голд»","#3 · @belleyou_ru · Мода и красотаНежная история Дарья Клюкина в уютных и нежных образах из новой капсулы Warm Hug. Легинсы, велосипедки и топы в нежных пастельных цветах. Капсула дополнена объемным кардиганом из пряжи букле с добавлением шерсти альпака. Вдохновляйтесь образами Дарьи и выбирайте свои модели для зимнего сезона. Капсула уже в магазинах и на belleyou.ru"
"#4 · @kapsulaready · Мода и красотаХорошая идея - дополнить образ платком, считает стилист vavdeyukekaterinaНаверняка после долгой зимы вы позабыли, что это крутой аксессуар. Пора доставать.","#4 · @localdress · Мода и красотаЛетняя капсула! Делимся ссылками на самые актуальные и трендовые модели этого сезона Ссылки: рубашка, майка, жилет, кардиган, джинсовые бермуды, шорты бойфренд, юбка-шорты, туфли, сандалии, носки, сумка плетеная, сумка велюровая, косынка, очки, бантик, колье, чокер, серьга, кольца"
"#5 · @ne_vasha_natasha_WB · ПродажиВСТРЕЧАЙТЕ: та самая трендовая жилетка-авиатор в стиле ZARA,но,естественно,с WILDBERRIES меня в размере SИ,кстати,я ещё немного поработала над новым форматом ЧТО СКАЖЕТЕ??","#5 · @look_web · Мода и красотаСеткаСетка станет в вашем образе, скорее, актуальным дополнением, чем ведущим предметом гардероба, но точно будет многофункциональной вещью, которая пригодится как в городе, так и во время отпуска. На пляже носите длинное полупрозрачное платье с базовым купальником, а в городе сочетайте его с кроп-топом и классическими брюками или леггинсами."


In [29]:
# Освобождаем VRAM
for k in list(loaded_models.keys()):
    del loaded_models[k]
loaded_models.clear()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Память освобождена.')


Память освобождена.
